# Train-set model selection and JK+ test evaluation

This notebook recovers the historical 115/29 subject split, applies it by subject ID to the finalized `coordination_age/Data/X.csv` and `y.csv`, selects an architecture using train-only outer-LOOCV RMSE, and evaluates the selected RBF kernel-ridge procedure with 95% jackknife+ intervals on the 29 held-out test subjects.

The test outcomes are not used for model selection, hyperparameter tuning, conformal calibration, or interval construction in this run. They are joined only after all test predictions and intervals have been produced. Because these outcomes were inspected during earlier project work, the resulting test metrics remain a post-hoc audit rather than pristine external validation.


## Methodology

1. Load the recovered historical 115/29 membership from `Data/train_test_split.csv`.
2. Validate its subject IDs and original partition order against the project’s fixed 32-feature `X` and aligned age target.
3. Compare RBF kernel ridge, ElasticNet, Lasso, Ridge, OLS, and Random Forest with an 18-configuration grid by train-set outer LOOCV. Tuned procedures use shuffled five-fold CV inside each outer fold.
4. Select the minimum raw OOF RMSE architecture; report OOF RMSE, MAE, and R².
5. Retain the 115 train-only LOO KRR pipelines, their absolute OOF residuals, and the KRR pipeline tuned/refit on all 115 training subjects.
6. Use those train-only objects to predict the 29 test subjects and form 95% JK+ intervals. Save row-level results for honest test-set interval diagnostics.


In [ ]:
from __future__ import annotations

import json
import os
from pathlib import Path
import sys
import time
from tempfile import TemporaryDirectory
import warnings

warnings.filterwarnings('ignore', message='pkg_resources is deprecated as an API.*')
os.environ['PYTHONWARNINGS'] = 'ignore:pkg_resources is deprecated as an API:UserWarning'

from IPython.display import display
from joblib import Memory
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.base import clone
from sklearn.ensemble import RandomForestRegressor
from sklearn.exceptions import ConvergenceWarning
from sklearn.impute import SimpleImputer
from sklearn.linear_model import ElasticNet, Lasso, LinearRegression, Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import GridSearchCV, KFold, LeaveOneOut, ParameterGrid
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

SEED, INNER_FOLDS, N_JOBS = 42, 5, 6
ID_COLUMN, TARGET_COLUMN = 'SubjectNumber', 'AgeInYears'
SEX_COLUMN, KRR_NAME = 'Sex_0Female_1Male', 'RBF Kernel Ridge'
ALPHA = 0.05
np.random.seed(SEED)
warnings.filterwarnings('ignore', category=ConvergenceWarning)

def find_project_root(start: Path) -> Path:
    for candidate in (start.resolve(), *start.resolve().parents):
        if (candidate / 'Data' / 'X.csv').exists() and (candidate / 'scripts' / 'fit_jackknife_plus_krr.py').exists():
            return candidate
    raise FileNotFoundError('Could not locate the coordination_age project root.')

PROJECT_ROOT = find_project_root(Path.cwd())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.krr_pipeline import FoldwiseWinsorizer, YeoJohnsonByFeature, make_krr_spec
from scripts.fit_jackknife_plus_krr import fit_jackknife_plus_krr, save_jackknife_plus_artifacts
from scripts.infer_jackknife_plus_krr import predict_with_jackknife_plus

OUTPUT_DIR = PROJECT_ROOT / 'outputs' / 'train115_jkplus'
ARTIFACT_DIR = PROJECT_ROOT / 'artifacts' / 'train115_jackknife_plus_krr'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

X = pd.read_csv(PROJECT_ROOT / 'Data' / 'X.csv')
y = pd.read_csv(PROJECT_ROOT / 'Data' / 'y.csv')[TARGET_COLUMN]
metadata = json.loads((PROJECT_ROOT / 'Data' / 'modeling_input_metadata.json').read_text())
subject_order = pd.Index(metadata['subject_order'], name=ID_COLUMN)
X.index = subject_order
y.index = subject_order

split_table = pd.read_csv(PROJECT_ROOT / 'Data' / 'train_test_split.csv')
required_split_columns = [ID_COLUMN, 'production_row_position', 'partition', 'partition_position']
assert split_table.columns.tolist() == required_split_columns
assert sorted(split_table['production_row_position']) == list(range(len(subject_order)))
mapped_subjects = subject_order.take(split_table['production_row_position'].to_numpy(int))
assert np.array_equal(mapped_subjects, split_table[ID_COLUMN].to_numpy(int))
train_split = split_table.query("partition == 'train'").sort_values('partition_position')
test_split = split_table.query("partition == 'test'").sort_values('partition_position')
assert train_split['partition_position'].tolist() == list(range(115))
assert test_split['partition_position'].tolist() == list(range(29))
train_ids = pd.Index(train_split[ID_COLUMN], name=ID_COLUMN)
test_ids = pd.Index(test_split[ID_COLUMN], name=ID_COLUMN)
assert len(train_ids) == 115 and len(test_ids) == 29
assert train_ids.intersection(test_ids).empty
assert set(train_ids).union(test_ids) == set(subject_order)

X_train, y_train = X.loc[train_ids].copy(), y.loc[train_ids].copy()
X_test = X.loc[test_ids].copy()
display(pd.DataFrame([
    {'partition': 'train', 'subjects': len(X_train), 'features': X_train.shape[1], 'missing_cells': int(X_train.isna().sum().sum())},
    {'partition': 'test', 'subjects': len(X_test), 'features': X_test.shape[1], 'missing_cells': int(X_test.isna().sum().sum())},
]))


## Candidate procedures

Every procedure uses fold-local median imputation, 1%/99% winsorization, feature-wise Yeo–Johnson transformation, and standardization. KRR also standardizes the target. Random Forest is fixed at 200 trees and depth 7.


In [ ]:
from scripts.evaluate_model_candidates import candidate_specs as make_comparison_specs

candidate_specs = {
    name: spec for name, spec in make_comparison_specs(X_train.columns).items()
    if name != KRR_NAME
}
krr_grid = make_krr_spec(X_train.columns, sex_column=SEX_COLUMN)['param_grid']
candidate_table = pd.DataFrame([
    {'procedure': KRR_NAME, 'configurations': len(list(ParameterGrid(krr_grid))), 'inner_cv': '5-fold'},
    *[
        {'procedure': name, 'configurations': 1 if spec['grid'] is None else len(list(ParameterGrid(spec['grid']))), 'inner_cv': 'fixed' if spec['grid'] is None else '5-fold'}
        for name, spec in candidate_specs.items()
    ],
])
display(candidate_table)

def fit_candidate(spec, X_fit, y_fit, memory):
    estimator = clone(spec['estimator']).set_params(memory=memory)
    if spec['grid'] is None:
        return estimator.fit(X_fit, y_fit), {}, np.nan
    tuner = GridSearchCV(
        estimator, spec['grid'], scoring='neg_mean_squared_error',
        cv=KFold(INNER_FOLDS, shuffle=True, random_state=SEED),
        refit=True, n_jobs=N_JOBS, pre_dispatch=N_JOBS, error_score='raise',
    ).fit(X_fit, y_fit)
    fitted = tuner.best_estimator_.set_params(memory=None)
    params = {key: value.item() if isinstance(value, np.generic) else value for key, value in tuner.best_params_.items()}
    return fitted, params, float(np.sqrt(-tuner.best_score_))

def metric_table(predictions):
    rows = []
    for procedure, group in predictions.groupby('procedure', sort=False):
        observed = group['chronological_age_years'].to_numpy(float)
        predicted = group['oof_prediction_years'].to_numpy(float)
        rows.append({
            'procedure': procedure,
            'n': len(group),
            'rmse_years': float(np.sqrt(mean_squared_error(observed, predicted))),
            'mae_years': float(mean_absolute_error(observed, predicted)),
            'r_squared': float(r2_score(observed, predicted)),
        })
    result = pd.DataFrame(rows).sort_values(['rmse_years', 'mae_years']).reset_index(drop=True)
    result.insert(0, 'rmse_rank', np.arange(1, len(result) + 1))
    return result


## Train-only nested LOOCV and KRR JK+ fitting

The KRR JK+ fit supplies KRR’s honest train OOF predictions as well as the 115 LOO calibration models and the train-refitted point-prediction model. The remaining candidates are evaluated under the same outer LOOCV and five-fold inner tuning rule. No test outcome is accessed here.


In [ ]:
fit_started = time.perf_counter()
loo_artifact, train_krr_model, krr_residuals, krr_selections = fit_jackknife_plus_krr(
    X_train, y_train, sex_column=SEX_COLUMN, seed=SEED,
    inner_folds=INNER_FOLDS, n_jobs=N_JOBS, progress_every=10,
)
artifact_paths = save_jackknife_plus_artifacts(
    loo_artifact=loo_artifact, production_model=train_krr_model,
    residuals=krr_residuals, selections=krr_selections, output_dir=ARTIFACT_DIR,
)
krr_predictions = pd.DataFrame({
    ID_COLUMN: krr_residuals['sample_id'].astype(int),
    'procedure': KRR_NAME,
    'chronological_age_years': krr_residuals['observed_y'].to_numpy(float),
    'oof_prediction_years': krr_residuals['loo_prediction'].to_numpy(float),
})
print(f'Saved {len(loo_artifact["loo_models"]) + 1} train-only KRR models in {(time.perf_counter() - fit_started) / 60:.2f} minutes.')


In [ ]:
prediction_records, selection_records = [], []
started = time.perf_counter()
for fold, (fit_idx, held_idx) in enumerate(LeaveOneOut().split(X_train), start=1):
    X_fit, y_fit = X_train.iloc[fit_idx], y_train.iloc[fit_idx]
    X_held = X_train.iloc[held_idx]
    position = int(held_idx[0])
    subject, observed = int(X_train.index[position]), float(y_train.iloc[position])
    with TemporaryDirectory(prefix='coordination_train115_candidates_') as cache_dir:
        memory = Memory(cache_dir, verbose=0)
        for procedure, spec in candidate_specs.items():
            fitted, params, inner_rmse = fit_candidate(spec, X_fit, y_fit, memory)
            prediction = float(np.asarray(fitted.predict(X_held)).ravel()[0])
            prediction_records.append({
                ID_COLUMN: subject, 'procedure': procedure,
                'chronological_age_years': observed, 'oof_prediction_years': prediction,
            })
            selection_records.append({
                'outer_fold': fold, ID_COLUMN: subject, 'procedure': procedure,
                'inner_cv_rmse_years': inner_rmse, 'best_parameters': repr(params),
            })
    if fold == 1 or fold % 10 == 0 or fold == len(X_train):
        elapsed = time.perf_counter() - started
        eta = elapsed / fold * (len(X_train) - fold)
        print(f'Other candidates {fold:3d}/{len(X_train)}; elapsed={elapsed / 60:.1f} min; ETA={eta / 60:.1f} min', flush=True)

model_selection_predictions = pd.concat([krr_predictions, pd.DataFrame(prediction_records)], ignore_index=True)
model_selection_selections = pd.concat([
    krr_selections.query("fit_role == 'leave_one_out'").assign(procedure=KRR_NAME),
    pd.DataFrame(selection_records),
], ignore_index=True, sort=False)
model_selection_metrics = metric_table(model_selection_predictions)
selected_architecture = str(model_selection_metrics.iloc[0]['procedure'])
COMPARISON_DIR = PROJECT_ROOT / 'outputs/model_comparison'
COMPARISON_DIR.mkdir(parents=True, exist_ok=True)
model_selection_predictions.to_csv(COMPARISON_DIR / 'comparison_oof_predictions.csv', index=False)
model_selection_selections.to_csv(OUTPUT_DIR / 'train115_model_selection_selections.csv', index=False)
model_selection_metrics.to_csv(COMPARISON_DIR / 'comparison_metrics.csv', index=False)
(OUTPUT_DIR / 'train115_selected_architecture.json').write_text(json.dumps({
    'selected_architecture': selected_architecture,
    'criterion': 'minimum train-set outer-LOOCV RMSE',
    'test_outcomes_used': False,
}, indent=2))
print('Selected architecture:', selected_architecture)
display(model_selection_metrics.round(4))
if selected_architecture != KRR_NAME:
    raise RuntimeError(f'KRR was not selected on train-only OOF RMSE; selected {selected_architecture!r}.')


## Held-out test predictions and 95% JK+ intervals

First produce predictions and intervals from the frozen train-only artifacts. Only afterward join the 29 chronological ages to calculate point-prediction error, empirical coverage, and interval-width diagnostics.


In [ ]:
test_intervals = predict_with_jackknife_plus(
    X_test, loo_artifact=loo_artifact, production_model=train_krr_model,
    residuals=krr_residuals, alpha=ALPHA,
).rename(columns={'prediction': 'predicted_age_years'})

# Test outcomes enter only after predictions and intervals are fixed.
y_test = y.loc[test_ids].copy()
test_results = test_intervals.copy()
test_results.insert(0, 'chronological_age_years', y_test)
test_results['residual_years'] = test_results['chronological_age_years'] - test_results['predicted_age_years']
test_results['absolute_error_years'] = test_results['residual_years'].abs()
test_results['covered'] = (
    (test_results['chronological_age_years'] >= test_results['jkplus_lower'])
    & (test_results['chronological_age_years'] <= test_results['jkplus_upper'])
)
test_results = test_results.reset_index()
test_results.to_csv(OUTPUT_DIR / 'train115_jkplus_test29_predictions.csv', index=False)

test_metrics = pd.Series({
    'n_test_subjects': len(test_results),
    'rmse_years': float(np.sqrt(mean_squared_error(test_results['chronological_age_years'], test_results['predicted_age_years']))),
    'mae_years': float(mean_absolute_error(test_results['chronological_age_years'], test_results['predicted_age_years'])),
    'r_squared': float(r2_score(test_results['chronological_age_years'], test_results['predicted_age_years'])),
    'empirical_coverage': float(test_results['covered'].mean()),
    'mean_interval_width_years': float(test_results['jkplus_width'].mean()),
    'median_interval_width_years': float(test_results['jkplus_width'].median()),
    'minimum_interval_width_years': float(test_results['jkplus_width'].min()),
    'maximum_interval_width_years': float(test_results['jkplus_width'].max()),
}, name='value')
test_metrics.rename_axis('metric').reset_index().to_csv(OUTPUT_DIR / 'train115_jkplus_test29_metrics.csv', index=False)
(OUTPUT_DIR / 'train115_jkplus_rank.json').write_text(json.dumps(test_intervals.attrs['jackknife_plus_rank'], indent=2))
display(test_metrics.to_frame().round(4))
display(test_results.head())


In [ ]:
# The standalone figure generator uses these saved predictions after fitting.
# Run scripts/visualizations/create_all_figures.py to rebuild the final figures.
display(test_results.head())


## Interpretation

- Architecture and hyperparameters were selected using only the 115 training subjects.
- The KRR artifact directory contains 115 train-only LOO pipelines plus one KRR pipeline tuned/refit on all 115 training subjects.
- The JK+ construction treats the KRR architecture and tuning grid as frozen. If model-family selection is reopened, that selection must be rerun inside each leave-one-out fit (or separated using independent data) to retain the formal guarantee for the expanded learning procedure.
- The 29 test intervals are generated entirely from train-only objects in this run. Their empirical coverage is a useful but imprecise post-hoc diagnostic because the test outcomes were accessed earlier in the project.
- `train115_jkplus_test29_predictions.csv` is the analysis-ready source for interval width, coverage, age-stratified, and calibration plots.
- These train-only artifacts are separate from the 144-subject production artifacts in the `coordination_age` repository.


## Rerun individual candidates

The final random forest uses 200 trees and 18 configurations: depth
`[4, 7, None]`, minimum leaf size `[1, 3, 5]`, and feature fraction `[0.5, 1.0]`.
To rerun just that candidate from the repository root:

```bash
python scripts/evaluate_model_candidates.py --models "Random Forest" --n-jobs 6
python scripts/visualizations/create_all_figures.py
```

Completed outer folds are reused only when their data, grid, code and software
match. A changed configuration requires a new `--output-dir`. Comparison tables
retain saved predictions for candidates not requested. This does not refit the
KRR production model or use test outcomes for tuning.
